In [3]:
import tarfile
import pandas as pd
from pathlib import Path

data_dir = Path("data")
extract_dir = data_dir / "booksummaries_extracted"

if not extract_dir.exists():
    with tarfile.open(data_dir / "booksummaries.tar.gz") as tar:
        tar.extractall(extract_dir)

columns = ["wikipedia_id", "freebase_id", "title", "author", "pub_date", "genres", "summary"]
df = pd.read_csv(
    extract_dir / "booksummaries" / "booksummaries.txt",
    sep="\t",
    names=columns,
)

In [ ]:
df = df[["title", "summary"]]

df.head()

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

example = tokenizer(df["summary"].iloc[0])
print(example["input_ids"][:30])
print(tokenizer.convert_ids_to_tokens(example["input_ids"][:30]))

[0, 3470, 5454, 6, 5, 793, 5276, 271, 15, 5, 24190, 6584, 6, 1519, 5, 3122, 15, 5, 3380, 13, 10, 529, 6, 147, 37, 12616, 5, 5868, 7, 37891]
['<s>', 'ĠOld', 'ĠMajor', ',', 'Ġthe', 'Ġold', 'Ġbo', 'ar', 'Ġon', 'Ġthe', 'ĠManor', 'ĠFarm', ',', 'Ġcalls', 'Ġthe', 'Ġanimals', 'Ġon', 'Ġthe', 'Ġfarm', 'Ġfor', 'Ġa', 'Ġmeeting', ',', 'Ġwhere', 'Ġhe', 'Ġcompares', 'Ġthe', 'Ġhumans', 'Ġto', 'Ġparasites']


In [6]:
df["summary_tokens"] = [len(ids) for ids in tokenizer(df["summary"].tolist())["input_ids"]]

df["summary_tokens"].describe()

count    16559.000000
mean       551.656622
std        644.783596
min          4.000000
25%        154.000000
50%        337.000000
75%        728.000000
max      13878.000000
Name: summary_tokens, dtype: float64

In [7]:
n_before = len(df)
df = df[df["summary_tokens"] <= 1024].reset_index(drop=True)
pct_dropped = (n_before - len(df)) / n_before * 100

print(f"Dropped {n_before - len(df)} of {n_before} summaries ({pct_dropped:.1f}%)")

Dropped 2476 of 16559 summaries (15.0%)


In [8]:
df["title_tokens"] = [len(ids) for ids in tokenizer(df["title"].tolist())["input_ids"]]

print("Max title tokens:", df["title_tokens"].max())
df["title_tokens"].describe()

Max title tokens: 57


count    14083.000000
mean         6.648299
std          2.549928
min          3.000000
25%          5.000000
50%          6.000000
75%          8.000000
max         57.000000
Name: title_tokens, dtype: float64

In [9]:
df["title_tokens"].value_counts().sort_index()

title_tokens
3      172
4     1851
5     3153
6     2939
7     2265
8     1436
9      843
10     506
11     304
12     189
13     138
14      81
15      45
16      32
17      32
18      25
19      18
20      14
21      10
22      12
23       5
24       3
26       3
27       4
28       1
44       1
57       1
Name: count, dtype: int64

In [10]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 2143.20it/s]


In [11]:
inputs = tokenizer(df["summary"].iloc[0], return_tensors="pt")
inputs["input_ids"].shape

torch.Size([1, 942])

In [12]:
output_ids = model.generate(**inputs, max_new_tokens=20)
output_ids

tensor([[    2,     0,    20,  5808,  8593,  2629,    14,   980,   198,     5,
         36713,  4846,    16,  6408,    88, 28289,  4063, 13171,   373,   525,
             2]])

In [ ]:
print("Generated:", tokenizer.decode(output_ids[0], skip_special_tokens=True))
print("Actual title:", df["title"].iloc[0])

print("Original summary:", df["summary"].iloc[0])

Generated:  The novel posits that space around the Milky Way is divided into concentric layers called Z
Actual title: A Fire Upon the Deep
Summary  The novel posits that space around the Milky Way is divided into concentric layers called Zones, each being constrained by different laws of physics and each allowing for different degrees of biological and technological advancement. The innermost, the "Unthinking Depths", surrounds the galactic core and is incapable of supporting advanced life forms at all. The next layer, the "Slow Zone", is roughly equivalent to the real world in behavior and potential. Further out, the zone named the "Beyond" can support futuristic technologies such as AI and FTL travel. The outermost zone, the "Transcend", contains most of the galactic halo and is populated by incomprehensibly vast and powerful posthuman entities. A human expedition investigates a five-billion-year-old data archive that offers the possibility of unimaginable riches for the ambitious yo

In [ ]:
shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(shuffled)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train = shuffled.iloc[:train_end]
val = shuffled.iloc[train_end:val_end]
test = shuffled.iloc[val_end:]

print(len(train), len(val), len(test), len(train) + len(val) + len(test))

In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)
train.to_csv("data/processed/train.csv", index=False)
val.to_csv("data/processed/val.csv", index=False)
test.to_csv("data/processed/test.csv", index=False)